In [ ]:
try:
    import cabana
except ImportError:
    %pip install -q git+https://github.com/miniscope/CaBaNA.git
    import cabana

# CaBaNA: one framework for place cells, social behavior and syllables

**Question this notebook answers:** can place-cell analysis and social-behavior analysis really share one framework?

The claim is that, once behavior is written in a common form, the *middle* of the analysis is identical:

- the **same metric code** (information content, AUROC, stability, population-vector overlap) runs on a 2D spatial grid, on scored social bouts, and on behavioral syllables;
- the **same circular-shift engine** gives a null distribution for each of them.

To check that this works we use a simulated session where we *know* which cells are tuned to what, then see whether the pipeline finds them (and does not find the others).

**Second half of the notebook:** the same session and the same pipeline machinery also give peri-event time histograms, mean activity per state, correlation with speed, population PCA and decoding on re-binned data, a GLM, overlap between cell classes, response clusters, and a cohort check. Changing the null distribution is a one-line change.

## The three-array idea

Every analysis here reduces to three arrays on the neural clock:

| object | shape | meaning |
| --- | --- | --- |
| **activity** | `(n_cells, n_time)` | deconvolved calcium, dF/F, or binned spikes |
| **states** | `(n_time, n_states)` bool | which behavioral state(s) the animal is in at each frame |
| **mask** | `(n_time,)` bool | which frames count at all (speed threshold, "was this stretch scored?") |

- A **spatial bin** is a state ("the animal is in bin 17"). A 2D arena is just a state space with a grid topology that remembers which bins are neighbors (needed for smoothing and for drawing maps).
- A **social bout** is a state ("sniffing"). States can overlap, because an animal can sniff while chasing.
- A **syllable** is a state ("still", "run", ...). Syllables are exclusive: exactly one per frame.

Metrics only ever see these three arrays. They never ask "is this space, or social behavior?".

## 1. Simulate a session with planted ground truth

`simulate_session` builds a 2D arena trajectory (with stationary pauses), scored social bouts (sniff, groom, chase, huddle), and kinematic syllables, and plants four disjoint cell populations: **place cells**, **social cells** (half excited, half inhibited), **syllable cells**, and **untuned cells** that fire at baseline only. The answer key lives in `session.meta["ground_truth"]`.

In [ ]:
import time
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from cabana import AnalysisSpec, Pipeline, ShuffleConfig, simulate_session

N_SHUFFLES = 500
session = simulate_session(seed=0)
gt = session.meta["ground_truth"]
sim = session.meta["simulation"]
social = gt["social_cells"]
n_exc = sum(d["sign"] == "excited" for d in social)
print(
    f"{sim['n_cells']} cells, {sim['duration_s']:.0f} s at {sim['fs']:g} Hz. Planted: "
    f"{len(gt['place_cells'])} place cells, {len(social)} social cells "
    f"({n_exc} excited / {len(social) - n_exc} inhibited), "
    f"{len(gt['syllable_cells'])} syllable cells, {len(gt['untuned_cells'])} untuned cells."
)
print("example social cells:", social[:4])

## 2. One pipeline

A `Pipeline` is three ordered lists: **transforms** (raw behavior to states), **preprocess** steps (masks, smoothing), and **analyses** (metric x states x optional shuffle). Each step is a small registered plugin whose parameters are its fields, so the whole pipeline can be written to YAML.

### What each transform does

| transform | what it does |
| --- | --- |
| `resample_signal(source="position")` | Behavior is tracked on its own clock (here 30 Hz, offset from the neural clock). This interpolates the x/y signals onto the neural frames. |
| `speed()` | Computes per-frame speed from x/y over a short window. |
| `bin_grid(signals=["x","y"], bin_size=20)` | Cuts the arena into 20 mm squares and makes each square a state. Output: a *spatial* state space with grid topology. |
| `bouts_to_states(source="social")` | Turns scored bouts (start, stop, label) into a boolean state matrix, one column per behavior, overlaps allowed. Also writes a `social_scored` mask. |
| `labels_to_states(source="syllables")` | Turns a per-frame label series into an exclusive categorical state space. Also writes a `syllables_covered` mask. |

### Why unscored is not rest, and why masks are per-analysis

Social bouts are only scored inside a window (here 30 s after the start to 30 s before the end). Frames outside it, or frames with no bout inside it, are **not** "rest" automatically: the experimenter may simply not have scored them. So `bouts_to_states` does not invent a rest state, and `social_scored` marks which frames the scoring actually covers.

Masks are chosen **per analysis** with `AnalysisSpec(masks=[...])` because different questions need different frames:

- **Place cells:** keep only frames where the animal is moving (`speed_threshold`, 20 mm/s), the standard way to remove immobility-related activity from spatial maps.
- **Social behavior:** do *not* apply the speed mask. Huddling is stationary, so a speed filter would throw away exactly the behavior of interest. Use only `social_scored`.
- **Syllables:** use no mask at all. "still" is one of the syllables we want to keep.

(`masks=None` would AND together *every* mask, which is rarely what you want when behaviors differ.)

In [ ]:
shuf = ShuffleConfig(n_shuffles=N_SHUFFLES, min_shift_s=20.0, seed=0)

pipe = Pipeline(
    transforms=[
        {"name": "resample_signal", "source": "position"},
        {"name": "speed"},
        # fixed arena extent: the same grid in every session (needed later for cross-session maps)
        {"name": "bin_grid", "signals": ["x", "y"], "bin_size": 20.0, "output": "position",
         "extent": [(0.0, 400.0), (0.0, 400.0)]},
        {"name": "bouts_to_states", "source": "social", "output": "social"},
        {"name": "labels_to_states", "source": "syllables", "output": "syllables"},
    ],
    preprocess=[{"name": "speed_threshold", "min_speed": 20.0}],
    analyses=[
        # spatial: same functions as syllables below, applied to grid bins
        AnalysisSpec(metric={"name": "information_content", "smooth_sigma_bins": 1.0},
                     states="position", shuffle=shuf, masks=["speed"]),
        AnalysisSpec(metric={"name": "split_half_stability", "smooth_sigma_bins": 1.0},
                     states="position", shuffle=shuf, masks=["speed"]),
        AnalysisSpec(metric={"name": "pv_stability"}, states="position",
                     shuffle=shuf, masks=["speed"]),
        # social: NOT speed-filtered, huddling is stationary
        AnalysisSpec(metric={"name": "auroc"}, states="social",
                     shuffle=shuf, masks=["social_scored"]),
        AnalysisSpec(metric={"name": "population_vector_overlap"}, states="social",
                     shuffle=None, masks=["social_scored"]),
        # syllables: no mask, 'still' is a syllable we want to keep
        AnalysisSpec(metric={"name": "information_content"}, states="syllables",
                     shuffle=shuf, masks=[]),
        AnalysisSpec(metric={"name": "auroc"}, states="syllables",
                     shuffle=shuf, masks=[]),
    ],
)

t0 = time.perf_counter()
session, results = pipe.run(session, progress=True)
print(f"\nPipeline runtime: {time.perf_counter() - t0:.1f} s")

### The same metric on space and on syllables

Look at the analysis list above: `information_content` appears on `position` (grid bins) *and* on `syllables`, and `auroc` appears on `social` *and* on `syllables`. It is literally the same class and the same `compute` function. Skaggs information is just "how much does knowing the state tell you about the cell's firing?", whether the state is a 20 mm square, a syllable, or a social behavior. The only thing that differs is the state space handed in: spatial grids additionally carry a topology, which allows optional smoothing (`smooth_sigma_bins`) and map plotting.

### How the circular-shift null works

To ask whether a cell's relationship to behavior is more than chance, we need a null that keeps everything *except* the alignment: the cell's firing statistics (bursting, slow drifts, rate) and the behavior's statistics (bout durations, autocorrelation) stay intact. The engine therefore:

1. rolls each cell's whole activity trace by a random offset (at least `min_shift_s` = 20 s, so no shuffle is nearly aligned with the data), wrapping around the end;
2. recomputes the metric against the **unshifted** states and mask;
3. repeats N times; the p-value is `(count of shuffles at least as extreme + 1) / (N + 1)`.

**Why shifting happens in wall-clock time before masking.** The shift is applied to the full recording, and only then are masked frames dropped. If instead we first dropped masked frames and shuffled the remainder, we would be shuffling a stitched-together trace in which distant moments have been made adjacent, and the null would no longer preserve the real temporal structure of the activity. Shifting first keeps the cell's own time course intact, and the mask then removes exactly the same frames it removes for the real data.

Tails are chosen by the metric: upper for information content (it cannot be negative, we ask for *more* than chance), two-sided for AUROC (a cell can be excited *or* inhibited).

## 3. Behavior overview

Trajectory colored by time, speed (grey bands are frames excluded by the 20 mm/s speed mask), the social ethogram (rows are behaviors, overlaps allowed), and the syllable strip.

In [ ]:
import cabana
fig = cabana.plots.plot_behavior_overview(session)
plt.show()

## 4. Place cells

Rate maps of the 12 place cells with the highest information, then six untuned cells for contrast. Each title shows the information value and its shuffle p-value. Place cells should show a compact bright field; untuned cells show structureless noise (their maps look blotchy but nothing is above the shuffle null).

In [ ]:
ic_pos = results.get("information_content", "position")
place = np.array(gt["place_cells"])
untuned = np.array(gt["untuned_cells"])
top = place[np.argsort(-ic_pos.values[place])[:12]]
fig = cabana.plots.plot_rate_maps(session, ic_pos, list(top))
plt.show()
fig = cabana.plots.plot_rate_maps(session, ic_pos, list(untuned[:6]))
plt.show()

### Reading a shuffle distribution

Grey histogram: the metric recomputed on every circularly shifted copy of the cell's activity. Dashed lines: the null bounds a value must cross to be called significant. Red line: the real value. Left, a place cell is far outside its own null. Right, an untuned cell sits inside it: its information value is nonzero (information is always positive in finite data) but unremarkable.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), layout="constrained")
cabana.plots.plot_shuffle_distribution(ic_pos, int(top[0]), ax=axes[0])
cabana.plots.plot_shuffle_distribution(ic_pos, int(untuned[0]), ax=axes[1])
plt.show()

### Information vs. stability

Two different questions about the same cells: *how much* spatial information does the cell carry, and *does the same map appear in both halves of the session?* Real place cells are high on both. Points are colored by which tests each cell passed. Cells with high information but low stability would be suspect.

In [ ]:
sh_pos = results.get("split_half_stability", "position")
fig = cabana.plots.plot_metric_scatter(ic_pos, sh_pos)
plt.show()

## 5. Social behavior

Each row of the heatmap is a cell, each column a behavior, color is AUROC (0.5 = no preference, red = more active during the behavior, blue = less active). Rows are sorted by their preferred behavior. White dots mark shuffle-significant cell/behavior pairs. Because AUROC compares in-state to out-of-state frames, it works with overlapping bouts and needs no "rest" state.

In [ ]:
au_soc = results.get("auroc", "social")
fig = cabana.plots.plot_tuning_heatmap(au_soc, sort_by="peak")
plt.show()

**Population-vector overlap** asks a different question: how similar is the *whole population's* activity pattern in one behavior to another? Diagonal is 1 by construction; off-diagonal values show which behaviors recruit similar vs. distinct ensembles. (No shuffle here: the overlap matrix is descriptive.)

In [ ]:
pvo = results.get("population_vector_overlap", "social")
fig = cabana.plots.plot_population_matrix(pvo, vmin=-1, vmax=1)
plt.show()

## 6. Syllables

Exactly the same plot function and the same AUROC code as the social heatmap, now with six kinematic syllables as the states.

In [ ]:
au_syl = results.get("auroc", "syllables")
ic_syl = results.get("information_content", "syllables")
fig = cabana.plots.plot_tuning_heatmap(au_syl, sort_by="peak")
plt.show()

## 7. Did we recover the truth?

For each planted population the bar shows the fraction of its cells called significant. The **untuned** bars are the false-positive rate and should sit near the dashed alpha line (or somewhat above it for multi-state results such as AUROC, where a cell counts as significant if *any* of its states is, so about 4 tests per cell for social and 6 for syllables).

One bar deserves a second look: a large fraction of **place** cells also come out significant for information content and AUROC on **syllables**. That is not a bug in the metrics, it is a real confound in the data. The simulated place cells only fire when the animal moves faster than 20 mm/s, and syllables such as `still`, `walk` and `run` are defined by speed, so a cell with a place field *is* informative about syllables. Real place cells are speed-modulated too. This is exactly the kind of thing a shared framework surfaces: the same cells, the same activity, two behavioral descriptions, and the analysis tells you they are not independent. Deciding how to disentangle them (speed as a covariate, matched-speed subsets, a joint state space) is an experimental question, not a pipeline one.

In [ ]:
fig = cabana.plots.plot_ground_truth_recovery(session, results)
summary = fig.cabana_summary
plt.show()
summary

A more specific view: for each analysis, how many *relevant* cells are detected, what is the false-positive rate in untuned cells, and, for AUROC, do cells respond in **their own planted state**, with the **right sign** (AUROC > 0.5 for excited, < 0.5 for inhibited)?

In [ ]:
pops = {
    "position": ("place", list(gt["place_cells"])),
    "social": ("social", [d["cell"] for d in social]),
    "syllables": ("syllable", [d["cell"] for d in gt["syllable_cells"]]),
}
rows = []
for r in results:
    if r.level != "cell" or r.shuffle is None:
        continue
    sig = np.asarray(r.shuffle.p_value) < 0.05
    any_sig = sig if sig.ndim == 1 else sig.any(axis=1)
    pop, cells = pops[r.target]
    row = {"metric": r.metric, "target": r.target, "relevant": pop,
           "frac_sig_relevant": any_sig[cells].mean(),
           "frac_sig_untuned (FPR)": any_sig[list(gt["untuned_cells"])].mean(),
           "frac_sig_own_state": np.nan, "sign_match": np.nan}
    if r.metric == "auroc" and r.target == "social":
        ci = np.array([d["cell"] for d in social])
        own = [r.state_names.index(d["behavior"]) for d in social]
        auc = r.values[ci, own]
        excited = np.array([d["sign"] == "excited" for d in social])
        ok = np.where(excited, auc > 0.5, auc < 0.5)
        row["frac_sig_own_state"] = sig[ci, own].mean()
        row["sign_match"] = ok[sig[ci, own]].mean()
    rows.append(row)
pd.DataFrame(rows).round(2)

## 8. More questions about the same session

So far every analysis asked "is this cell tuned to a *state*?". Researchers also ask other things of the same data. None of them needs new loaders or a new session object: they are more `AnalysisSpec`s on the arrays we already have.

| question | analysis | target |
| --- | --- | --- |
| How does a cell respond around the *onset* of a behavior? | `peth` | a state space |
| What is the cell's mean activity in each behavior? | `mean_per_state` | a state space |
| Does the cell track a *continuous* variable such as speed? | `signal_correlation` | a signal (`signal="speed"`) |

Notice `AnalysisSpec` takes exactly one of `states=` (a state space) or `signal=` (a continuous per-frame variable). All three use the **same circular-shift null** as before. The PETH is a linear function of activity, so its null is computed exactly and fast.

PETH settings follow the survey (`docs/ANALYSIS_COVERAGE.md`): keep bouts at least 1.5 s long and at least 1 s after the previous bout, a window of -4 to +4 s around onset, baseline -4 to -1 s, response 0 to 1.5 s. The tested statistic is the mean response minus the mean baseline across events.

In [ ]:
shuf = ShuffleConfig(n_shuffles=N_SHUFFLES, min_shift_s=20.0, seed=0)
extra = Pipeline(analyses=[
    AnalysisSpec(metric={"name": "mean_per_state", "statistic": "difference"},
                 states="social", shuffle=shuf, masks=["social_scored"]),
    AnalysisSpec(metric={"name": "peth", "pre_s": 4.0, "post_s": 4.0,
                         "baseline_s": (-4.0, -1.0), "response_s": (0.0, 1.5),
                         "min_duration_s": 1.5, "min_gap_s": 1.0},
                 states="social", shuffle=shuf, masks=["social_scored"]),
    AnalysisSpec(metric={"name": "signal_correlation", "max_lag_s": 2.0},
                 signal="speed", shuffle=shuf, masks=[]),
])
# the session already holds the transformed states, so only the analyses run here
session, extra_results = extra.run(session, progress=True)
for r in extra_results:
    results.add(r)
print(len(results), "results in one ResultSet")

### Peri-event time histograms

Each panel is one behavior. Rows are cells, columns are time relative to bout onset (dashed line), color is the baseline-subtracted event average (red = above baseline). Stars at the right edge mark cells significant under the shuffle test, **not** the Wilcoxon p in the extras: events within one animal are autocorrelated, so the shuffle p is the one to use. The lower trace averages all cells (grey) and the significant cells (orange). Grey and orange bands are the baseline and response windows.

In [ ]:
pe = results.get("peth", "social")
print("events per behavior:", dict(zip(pe.state_names, pe.extras["n_events"])))
fig = cabana.plots.plot_peth(pe, session, ncols=4)
plt.show()

A single cell, event by event, is the sanity check behind the average: the top social cell (largest |PETH response| in its own planted behavior). The image shows every event, the trace below is their mean +/- SEM.

In [ ]:
own = [pe.state_names.index(d["behavior"]) for d in social]
ci_soc = np.array([d["cell"] for d in social])
best = int(np.argmax(np.abs(np.nan_to_num(pe.values[ci_soc, own]))))
top_cell, top_state = int(ci_soc[best]), social[best]["behavior"]
print("top social cell:", top_cell, top_state, social[best]["sign"])
fig = cabana.plots.plot_event_raster(session, pe, top_cell, top_state,
                                     mask=session.valid_mask(["social_scored"]))
plt.show()

### Mean activity per state

`statistic="difference"` reports mean activity inside the behavior minus mean outside it, so 0 means no preference and the sign says excited or inhibited (no need to know a cell's baseline rate). Same heatmap function as AUROC; here the color is the raw difference. AUROC and mean difference agree on who responds, but the mean difference keeps the *size* of the effect in activity units, AUROC the *reliability*.

In [ ]:
fig = cabana.plots.plot_tuning_heatmap(results.get("mean_per_state", "social"), sort_by="peak")
plt.show()

### Correlation with speed (a continuous target)

This is both a question in its own right and the standard confound check: place cells in this simulation only fire above 20 mm/s, so they should correlate with speed, while untuned cells should not. The extras also hold the cross-correlation at lags up to +/-2 s (`xcorr`, `lags_s`, `best_lag_s`; a positive best lag means the cell *follows* speed).

Left: a place cell's observed correlation (red) against its own circular-shift null. Right: an untuned cell.

In [ ]:
sc = results.get("signal_correlation", "speed")
pc = int(place[np.nanargmax(np.abs(sc.values[place]))])
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), layout="constrained")
cabana.plots.plot_shuffle_distribution(sc, pc, ax=axes[0]); axes[0].set_title(f"place cell {pc}")
cabana.plots.plot_shuffle_distribution(sc, int(untuned[0]), ax=axes[1]); axes[1].set_title(f"untuned cell {int(untuned[0])}")
plt.show()
sig_sc = sc.shuffle.significant
print(f"significant speed correlation: place {sig_sc[place].mean():.2f}, untuned {sig_sc[untuned].mean():.2f}")

## 9. Population analyses on re-binned data

PCA, decoding and GLMs work on a coarser clock (100 ms to 1 s) than the 20 Hz frames. `rebin_time` is a preprocess step that resamples **everything on the neural clock together**: activity is summed (events) or averaged (dF/F), masks and states are resampled by occupancy fraction, signals are averaged. After it the session has fewer frames (`n_time` changes) and `fs` changes accordingly, but remains a normal session, so the *same* `AnalysisSpec` machinery applies.

This **second pipeline** runs on the session returned by the first one (so it already has the state spaces):

1. `rebin_time(bin_s=0.5)`
2. `design_matrix(states=["social"], signals=["speed"], basis=...)`: the regressors for the GLM. Each social state is convolved with 4 raised-cosine bumps spanning 1 s before to 3 s after the state (so a response that *follows* or *anticipates* the behavior can be captured), and speed enters as a standardized covariate.

Analyses:

- `pca_population`: distances between the mean population vectors of each social state (in z-scored population space).
- `decode_states`: a cross-validated linear decoder of each behavior from the population. **Folds are contiguous blocks of time**, never random frames, because neighboring frames are nearly identical; random folds would leak and inflate accuracy.
- `glm_encoding`: a Poisson regression per cell on the design matrix, again block cross-validated. For each regressor *group* (one behavior, or speed) it reports the **unique contribution**: how much held-out deviance explained is lost when that group is removed.

### Why a different null here

Shifting each cell's activity separately is the right null for single-cell tuning. For population questions the question is about the *labels*: do the behavior labels carry population structure beyond what a label sequence with the same autocorrelation would give? So these analyses use `label_shuffle`, which permutes the behavior in blocks of `block_s` seconds while activity stays put. It is a one-line change in the `ShuffleConfig`: `null={"name": "label_shuffle", "block_s": 5.0}`.

In [ ]:
null = {"name": "label_shuffle", "block_s": 5.0}
pop_pipe = Pipeline(
    preprocess=[
        {"name": "rebin_time", "bin_s": 0.5},
        {"name": "design_matrix", "states": ["social"], "signals": ["speed"],
         "basis": {"n_basis": 4, "pre_s": 1.0, "post_s": 3.0}},
    ],
    analyses=[
        AnalysisSpec(metric={"name": "pca_population"}, states="social", masks=["social_scored"],
                     shuffle=ShuffleConfig(n_shuffles=100, null=null, seed=0)),
        AnalysisSpec(metric={"name": "decode_states"}, states="social", masks=["social_scored"],
                     shuffle=ShuffleConfig(n_shuffles=30, null=null, seed=0)),
        AnalysisSpec(metric={"name": "glm_encoding"}, signal="design", masks=["social_scored"],
                     shuffle=None),
    ],
)
t0 = time.perf_counter()
binned, pop_results = pop_pipe.run(session, progress=True)
print(f"{session.n_time} frames at {session.fs:g} Hz -> {binned.n_time} frames at {binned.fs:g} Hz; "
      f"{time.perf_counter() - t0:.1f} s")
print("design columns:", cabana.design_columns(binned)[:6], "...", len(cabana.design_columns(binned)), "total")

### PCA: how far apart are the behaviors in population space?

Matrix of pairwise distances between the mean population vectors of the social states. A large entry means the whole population is in a clearly different pattern in the two behaviors. Whether these distances are *bigger than expected for arbitrary labels* is what the label-shuffle null (stored in `pca.shuffle`) answers.

In [ ]:
pca = pop_results.get("pca_population", "social")
fig = cabana.plots.plot_population_matrix(pca, cmap="magma")
plt.show()

### Decoding against its null

Bars: balanced accuracy of a one-vs-rest decoder per behavior on held-out time blocks (chance = 0.5). Black ticks: the 95th percentile of the accuracy obtained with block-shuffled labels. A bar above its tick means the population carries information about that behavior beyond label autocorrelation.

In [ ]:
dec = pop_results.get("decode_states", "social")
fig, ax = plt.subplots(figsize=(5.5, 3.8), layout="constrained")
xs = np.arange(len(dec.state_names))
ax.bar(xs, dec.values, color="C0", label="decoder (held-out blocks)")
ax.plot(xs, dec.shuffle.null_hi, "k_", ms=28, mew=2, label="null 95th pct (label_shuffle)")
ax.axhline(0.5, color="0.6", ls="--", lw=1)
ax.set_xticks(xs, dec.state_names); ax.set_ylim(0.4, 1.0)
ax.set_ylabel("balanced accuracy (one vs rest)"); ax.legend(fontsize=8)
plt.show()
pd.DataFrame({"state": dec.state_names, "balanced_acc": dec.values,
              "null_hi": dec.shuffle.null_hi, "p": dec.shuffle.p_value}).round(3)

### GLM: which regressor does each cell need?

For each cell the GLM asks which behavior (or speed) is *uniquely* needed to predict it on held-out time. Below, for each planted social cell: is its top group its own behavior? And for place cells: is it speed? (The GLM sees speed and the social states together, so a cell is credited to speed only to the extent that speed adds information the behaviors do not.)

In [ ]:
glm = pop_results.get("glm_encoding", "design")
groups = [str(g) for g in glm.extras["group_names"]]
top = np.nanargmax(glm.values, axis=1)
want = np.array([groups.index(f"social:{d['behavior']}") for d in social])
print("regressor groups:", groups)
print(f"social cells whose top group is their own behavior: {np.mean(top[ci_soc] == want):.2f}")
print(f"place cells whose top group is speed:               {np.mean(top[place] == groups.index('speed')):.2f}")
de = glm.extras["deviance_explained"]
print(f"median held-out deviance explained: social {np.nanmedian(de[ci_soc]):.3f}, untuned {np.nanmedian(de[untuned]):.3f}")
fig, ax = plt.subplots(figsize=(6, 3.8), layout="constrained")
order = np.argsort(want, kind="stable")
im = ax.imshow(glm.values[ci_soc][order], aspect="auto", cmap="viridis")
ax.set_xticks(range(len(groups)), groups, rotation=45, ha="right")
ax.set_ylabel("planted social cells (sorted by own behavior)")
fig.colorbar(im, ax=ax, label="unique deviance explained")
plt.show()

## 10. Amplitude as a dimension: mutual information and Bayesian decoding

A rate map collapses the activity in each state to one number, its mean. But a cell can carry information in *how much* it fires, not only in *where*: discretise the amplitude into a few levels and the tuning curve becomes a **state x level histogram**, p(level | state). **Mutual information (MI)** between state and level is a single number over that histogram, and it picks up whatever the amplitude distribution says.

One thing to know before reading these histograms. If amplitude adds nothing beyond "the animal is in the field", the in-field columns are all *identical* to each other, but they are **not uniform**: deconvolved activity is dominated by zeros, so most of every column sits in the lowest level. So look at p(level | state) and at the **lift**, log2 of p(level | state) / p(level), which is zero wherever a state looks like the overall distribution. MI is the average of that lift.

In [ ]:
shuf_mi = ShuffleConfig(n_shuffles=200, min_shift_s=20.0, seed=0)
mi_spec = lambda **kw: AnalysisSpec(metric={"name": "mutual_information", **kw}, states="position",
                                    shuffle=shuf_mi, masks=["speed"])
t0 = time.perf_counter()
_, mi_res = Pipeline(analyses=[mi_spec()]).run(session)                       # 5 quantile levels
_, bin_res = Pipeline(analyses=[mi_spec(binning="binary")]).run(session)      # on / off only
mi_pos = mi_res.get("mutual_information", "position")
mi_bin = bin_res.get("mutual_information", "position")
print(f"{time.perf_counter() - t0:.1f} s")

sig = {"Skaggs information": ic_pos.shuffle.p_value < 0.05,
       "MI, 5 quantile levels": mi_pos.shuffle.p_value < 0.05,
       "MI, binary (on/off)": mi_bin.shuffle.p_value < 0.05}
pd.DataFrame({"detect_place": {k: v[place].mean() for k, v in sig.items()},
              "FPR_untuned": {k: v[untuned].mean() for k, v in sig.items()}}).round(2)

Two ways to look at the histogram of the three planted place cells with the most MI. Left to right within a row are the activity levels (lowest to highest); each map shows, for one level, how likely that level is at every location. **Conditional** shows p(level | location): the lowest level is everywhere, and the higher levels light up inside the field. **Lift** removes the overall level frequencies: red means the level is over-represented there, blue under-represented. An untuned cell (last row) has a flat lift map: its amplitude says nothing about location.

In [ ]:
top3 = place[np.argsort(-np.nan_to_num(mi_pos.values[place]))[:3]]
pos_states = session.states["position"]
fig = cabana.plots.plot_joint_histogram(mi_pos, list(top3), states=pos_states, normalize="conditional")
plt.show()
fig = cabana.plots.plot_joint_histogram(mi_pos, [*top3, int(untuned[0])], states=pos_states, normalize="lift")
plt.show()

### Skaggs information versus MI

Both are measured in bits and both are compared with the same circular-shift null, so each axis is shown minus its null mean. Place cells (black rings) rise on both; the question is whether MI finds cells that Skaggs information misses or the other way around. In this simulation neither finds cells the other misses: the planted cells change *where* they fire, not how strongly, so the extra amplitude detail adds no detections and, as the sweep below shows, costs power on short recordings.

In [ ]:
fig = cabana.plots.plot_information_comparison(
    ic_pos, mi_pos, labels=("Skaggs information (bits/event)", "mutual information (bits)"),
    highlight=list(place))
plt.show()

### Why this is safe: the null carries the same bias

With 400 location bins and several levels, the histogram has thousands of cells and a recording only fills a few of them, so the *observed* MI is biased upward. That does not produce false positives, because every circularly shifted copy of the cell has the same finite-sampling bias; the null is shifted up by the same amount. The price is power: more levels means fewer samples per histogram cell. The sweep below (reduced settings: 80 cells, 50 shuffles) varies the number of levels and the recording duration. Solid lines are MI, dashed is Skaggs information (which does not use levels, so it is flat). Read it as: the false-positive rate (right) should stay near the dashed 0.05 line for every setting, while detection (left) falls as levels grow and recovers with longer recordings.

In [ ]:
def levels_sweep(durations=(300.0, 600.0), levels=(2, 3, 5, 8), n_shuf=50):
    shuf = ShuffleConfig(n_shuffles=n_shuf, min_shift_s=20.0, seed=0)
    prep = Pipeline(transforms=pipe.transforms, preprocess=pipe.preprocess, analyses=[])
    rows = []
    for dur in durations:
        sess, _ = prep.run(simulate_session(duration_s=dur, n_cells=80, seed=0))
        g = sess.meta["ground_truth"]
        rel, unt = list(g["place_cells"]), list(g["untuned_cells"])
        runs = [("skaggs_information", None, {"name": "information_content", "smooth_sigma_bins": 1.0})]
        runs += [("mutual_information", k, {"name": "mutual_information", "n_levels": k}) for k in levels]
        for label, k, metric in runs:
            spec = AnalysisSpec(metric=metric, states="position", shuffle=shuf, masks=["speed"])
            _, res = Pipeline(analyses=[spec]).run(sess)
            p = np.asarray(res.get(metric["name"], "position").shuffle.p_value)
            for kk in ([k] if k else levels):    # Skaggs does not depend on levels: repeat as a flat line
                rows.append({"metric": label, "n_levels": kk, "duration_s": dur,
                             "detection": (p[rel] < 0.05).mean(), "fpr": (p[unt] < 0.05).mean()})
    return pd.DataFrame(rows)

t0 = time.perf_counter()
sweep = levels_sweep()
print(f"{time.perf_counter() - t0:.1f} s")
fig = cabana.plots.plot_level_sweep(sweep)
plt.show()

### The same histogram is a naive-Bayes likelihood table

Turn the question around: given the population's activity at one moment, *where is the animal*? A naive-Bayes decoder multiplies, over cells, the likelihood of each cell's current activity given each location, then picks the most probable location. The state x level histogram above is exactly such a likelihood table (the `empirical` option); the Gaussian option (the Bayesian decoder of Shuman et al. 2020, *Nat Neurosci*) and the Poisson option fit a parametric likelihood per location instead. Held-out blocks are decoded by a model that never saw them, and the null is the same `label_shuffle` used for population decoding earlier. Errors are in mm; chance is always guessing the most occupied bin.

In [ ]:
null = {"name": "label_shuffle", "block_s": 5.0}
dec_shuf = ShuffleConfig(n_shuffles=20, null=null, seed=0)
dec = {}
t0 = time.perf_counter()
for lik in ("gaussian", "poisson"):   # gaussian = Shuman 2020 (per-state sigma)
    spec = AnalysisSpec(metric={"name": "bayesian_decoder", "likelihood": lik, "smooth_sigma_bins": 1.0},
                        states="position", shuffle=dec_shuf, masks=["speed"])
    dec[lik] = Pipeline(analyses=[spec]).run(session)[1].get("bayesian_decoder", "position")
print(f"{time.perf_counter() - t0:.1f} s")
print(pd.DataFrame({k: {"median error (mm)": r.extras["error_overall"][0],
                        "chance error (mm)": r.extras["chance_error"][0],
                        "median p over states": np.nanmedian(r.shuffle.p_value)}
                    for k, r in dec.items()}).round(3).T)
fig = cabana.plots.plot_decoded_trajectory(dec["poisson"], session, pos_states)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6.2, 3.8), layout="constrained")
bins = np.linspace(0, np.nanpercentile(dec["gaussian"].extras["error"], 99.5), 50)
for i, (k, r) in enumerate(dec.items()):
    ax.hist(r.extras["error"], bins=bins, histtype="step", color=f"C{i}", density=True, label=k)
    ax.axvline(r.extras["error_overall"][0], color=f"C{i}", ls="--", lw=1)
ax.axvline(dec["gaussian"].extras["chance_error"][0], color="k", ls=":", label="chance")
ax.set_xlabel("decoding error (mm); dashed = median"); ax.set_ylabel("density of frames"); ax.legend(frameon=False)
plt.show()

Both decoders beat chance by a wide margin; Poisson does better because it scores a window of activity rather than one frame's value. The `empirical` likelihood is the data-sufficiency problem made concrete: with 400 bins and few positive frames per bin, two levels (silent / active) decode better than five, because the few positive frames get split across four levels, and a Dirichlet pseudocount of 1 (instead of the default 0.1) pushes the five-level decoder all the way to chance. Both trail Poisson. The decoder is not specific to space: on categorical states the value per behavior is **recall**. It needs one state per frame, so first make an exclusive version of the social states (overlaps resolved in the listed order, plus a `rest` state).

There is one trap. The Gaussian likelihood of Shuman et al. gives every (cell, state) its own width (`sigma="per_state"`). On a grid every bin has plenty of frames, but behaviors are unbalanced: a rare behavior has few training blocks, a noisy mean and hence a wide sigma, so its likelihood is nearly flat and it wins whenever a cell is active, while the common behaviors are crowded out. Below, per-state sigma scores about 0.06 accuracy on the default session. `sigma="pooled"` (one within-state sigma per cell, as in linear discriminant analysis) or the Poisson likelihood avoids it (about 0.42 and 0.50 measured on the default session); we use pooled here.

In [ ]:
session = Pipeline(transforms=[{"name": "bouts_to_states", "source": "social", "output": "social_exclusive",
                                "exclusive": True, "add_rest": True}]).run(session)[0]
spec = lambda sig, shuffle: AnalysisSpec(
    metric={"name": "bayesian_decoder", "sigma": sig, "min_frames_per_class": 20},
    states="social_exclusive", shuffle=shuffle, masks=["social_exclusive_scored"])
bd_soc = Pipeline(analyses=[spec("pooled", dec_shuf)]).run(session)[1].get("bayesian_decoder", "social_exclusive")
bd_ps = Pipeline(analyses=[spec("per_state", None)]).run(session)[1].get("bayesian_decoder", "social_exclusive")
print(f"overall accuracy: pooled {bd_soc.extras['accuracy_overall'][0]:.2f}, "
      f"per_state {bd_ps.extras['accuracy_overall'][0]:.2f}")
pd.DataFrame({"state": bd_soc.state_names, "recall": bd_soc.values,
              "null_hi": bd_soc.shuffle.null_hi, "p": bd_soc.shuffle.p_value}).round(3)

With pooled sigma the decoder beats its `label_shuffle` null in all five behaviors (recall 0.27 to 0.52 against null 95th percentiles of 0.21 to 0.25; overall accuracy 0.42, 0.06 with per-state sigma). With only 20 shuffles the smallest possible p is 1/21 = 0.048, so every behavior sits at the floor; raise `n_shuffles` for finer p-values. Recall is modest because only a small fraction of the population is tuned to each behavior and each frame is decoded on its own.

## 11. The choice of null is a one-line change

Everything above used `circular_shift` for single cells and `label_shuffle` for populations. Here is the **same AUROC analysis** on social behavior with three nulls. Only the `null=` entry of `ShuffleConfig` differs:

| null | what is kept | what is broken | typical use |
| --- | --- | --- | --- |
| `circular_shift` | both time series intact | the alignment, by sliding activity | default; single-cell tuning |
| `label_shuffle(block_s)` | activity intact | labels permuted in blocks of `block_s` | population analyses, decoders |
| `bout_shuffle` | activity intact; bout count and durations | bouts re-placed at random times | scored bouts (CaSA's null) |

We report, for each null, the fraction of planted social cells detected **in their own behavior**, and the false-positive rate of untuned cells **per state**. They should be similar here: the simulated data has no confounds that distinguish the nulls. With real data (slow drifts, bursty bouts) the nulls can disagree, and that disagreement is informative.

In [ ]:
N_NULL = 100
rows = []
for label, null in {"circular_shift": {"name": "circular_shift", "min_shift_s": 20.0},
                    "label_shuffle": {"name": "label_shuffle", "block_s": 2.0},
                    "bout_shuffle": {"name": "bout_shuffle"}}.items():
    p_ = Pipeline(analyses=[AnalysisSpec(
        metric={"name": "auroc"}, states="social", masks=["social_scored"],
        shuffle=ShuffleConfig(n_shuffles=N_NULL, null=null, seed=0))])   # <- the one line that changes
    _, res_ = p_.run(session)
    r = res_.get("auroc", "social")
    sig = np.asarray(r.shuffle.significant, bool)
    own_ = [r.state_names.index(d["behavior"]) for d in social]
    rows.append({"null": label,
                 "frac_social_detected_own_behavior": sig[ci_soc, own_].mean(),
                 "untuned_FPR_per_state": sig[untuned].mean()})
pd.DataFrame(rows).round(2)

## 12. Who responds to what: overlap and clusters

These functions work on **results**, not on activity, so they apply to any analysis.

**Overlap vs chance.** Are the place cells (information content significant) and the social-responsive cells (AUROC significant in any state) the same cells more often than independence would predict? `overlap_vs_chance` compares the observed overlap with the hypergeometric expectation (Fisher exact test). In this simulation the populations are planted as disjoint, so the honest expectation is *less* overlap than chance (odds ratio below 1); with real data an odds ratio above 1 would be the interesting finding (for example, multiplexed coding).

In [ ]:
from cabana import overlap_vs_chance, significant_cells, response_features, response_clusters
ic_pos = results.get("information_content", "position")
ov = overlap_vs_chance(significant_cells(ic_pos), significant_cells(au_soc))
print(ov)
fig = cabana.plots.plot_overlap(ov, labels=("place", "social"))
plt.show()

**Response clusters.** Instead of asking about one behavior at a time, describe each cell by its full response profile (here its AUROC and its mean difference in every social behavior), z-score the features and cluster. `k=None` tries k = 2..6 and keeps the best silhouette score. The heatmap shows each cluster's mean profile (row labels give cluster sizes); the right panel is the score for each k. The table compares clusters with the planted populations, which the clustering never saw. Unsupervised clusters rarely match the labels one to one: cells with the same behavior but opposite sign, or weak responders, can group differently.

In [ ]:
feats, fnames, kept = response_features(
    results, which=[("auroc", "social"), ("mean_per_state", "social")])
clusters = response_clusters(feats, k=None, k_range=(2, 6))
label = np.full(session.n_cells, "other", dtype=object)
label[place] = "place"; label[[d["cell"] for d in gt["syllable_cells"]]] = "syllable"
label[untuned] = "untuned"
for d in social:
    label[d["cell"]] = f"social_{d['sign']}"
print(f"k = {clusters.k}, silhouette by k: { {k: round(v, 2) for k, v in clusters.scores.items()} }")
fig = cabana.plots.plot_clusters(clusters, fnames)
plt.show()
pd.crosstab(pd.Series(clusters.labels, name="cluster"), pd.Series(label[kept], name="planted population"))

## 13. Across sessions and animals

Two small checks of the cohort tools, on short (300 s) simulated sessions so they run quickly. We run the **same pipeline** on two more sessions (seeds 1 and 2) and treat them as: `s0` and `s1` = animal A, `s2` = animal B. Statistics in this field treat the **animal**, not the cell, as the unit of replication, so `fraction_significant_by_animal` pools sessions and cells within animal.

Then a **negative control** for cross-session analysis. `cross_session_map_correlation` correlates each matched cell's rate map in two sessions, against a null that pairs the cell with *other* cells. Normally the `CellMatch` comes from a registration tool (CellReg); here `s0` and `s1` are different simulated animals, so we deliberately register cell *i* to cell *i* (an identity match, which is wrong). Unmatched cells should **not** correlate: median r near 0 and a population p-value that is not small. If this control came out significant, the analysis would be broken.

In [ ]:
from cabana import (CellMatch, cohort_table, fraction_significant_by_animal,
                    cross_session_map_correlation, remapping_summary)
by_session = {"s0": results}
other = {}
small_pipe = Pipeline(transforms=pipe.transforms, preprocess=pipe.preprocess,
                      analyses=[AnalysisSpec(metric={"name": "information_content", "smooth_sigma_bins": 1.0},
                                             states="position", shuffle=ShuffleConfig(n_shuffles=100, seed=0), masks=["speed"]),
                                AnalysisSpec(metric={"name": "auroc"}, states="social",
                                             shuffle=ShuffleConfig(n_shuffles=100, seed=0), masks=["social_scored"])])
for sid, seed in (("s1", 1), ("s2", 2)):
    other[sid], by_session[sid] = small_pipe.run(simulate_session(seed=seed, duration_s=300.0))
animal = {"s0": "A", "s1": "A", "s2": "B"}
table = cohort_table(by_session, animal)
fracs = fraction_significant_by_animal(table, metric="auroc", target="social")
display(fracs.round(2))
fig = cabana.plots.plot_cohort_fractions(fracs)
plt.show()

In [ ]:
ic_s1 = by_session["s1"].get("information_content", "position")
n_id = min(session.n_cells, other["s1"].n_cells)
ident = CellMatch(session_a="s0", session_b="s1", pairs=np.c_[np.arange(n_id), np.arange(n_id)])
cs = cross_session_map_correlation(ic_pos, ic_s1, ident, n_shuffles=200,
                                   states=(session.states["position"], other["s1"].states["position"]))
print(remapping_summary(cs))
fig = cabana.plots.plot_cross_session(cs)
plt.show()

## 14. Add your own metric

A metric is a pydantic class registered by name. It receives `(activity, states, mask, fs)` and returns an array whose first axis is cells. Every analysis above (including `mean_per_state`, `peth` and `glm_encoding`) is registered exactly this way. Below, the *fraction of frames with any activity* in each state, which the library does not ship. Because it implements only `compute`, the shuffle engine falls back to a plain loop; implementing `compute_shifted` (see `shifted_sums`) makes it fast but is optional.

In [ ]:
from typing import ClassVar
import numpy as np
from cabana import Metric, register  # (built-ins are registered the same way)

@register("metric")
class ActiveFraction(Metric):
    """Fraction of masked frames with any activity, in each state."""
    name: ClassVar[str] = "active_fraction"
    default_tail: ClassVar[str] = "two"

    def compute(self, activity, states, mask, fs):
        m = states.masked(mask) > 0                        # (n_time, n_states) bool
        out = np.full((activity.shape[0], m.shape[1]), np.nan)
        for s in range(m.shape[1]):
            if m[:, s].any():
                out[:, s] = (activity[:, m[:, s]] > 0).mean(axis=1)
        return out                                         # (n_cells, n_states)

custom = Pipeline(
    transforms=pipe.transforms, preprocess=pipe.preprocess,
    analyses=[AnalysisSpec(metric={"name": "active_fraction"}, states="syllables",
                           shuffle=ShuffleConfig(n_shuffles=100, seed=0), masks=[])],
)
_, custom_results = custom.run(session)
r = custom_results.get("active_fraction")
print("significant (cell, syllable) pairs:", int(r.shuffle.significant.sum()))
custom_results.to_dataframe().query("significant").head()

## 15. Save the run

The results table, the pipeline definition and the answer key are plain files. The pipeline YAML fully describes the analysis, so it can be re-run on another session.

In [ ]:
df = results.to_dataframe()
print(df.shape)
print(pipe.to_yaml()[:600])

## What's next

- **Real loaders:** Minian, CaImAn, DeepLabCut, BORIS and `.camap` bundles, so real sessions fill the same three arrays.
- **Registration:** `CellMatch` is a container; wiring CellReg output into it is the loader's job.
- **More analyses from the survey:** see `docs/ANALYSIS_COVERAGE.md` for what is implemented and what is not.
- **CaLab app:** CaBaNA is intended to become an app within [CaLab](https://github.com/miniscope/CaLab).

The point of the demo: the loader and the transform are where experiments differ. Everything in the middle (metrics, masks, nulls, plots) is shared.